# Quanto vale ser Superhost? Airbnb no Rio de Janeiro

**Autora:** Amanda de Souza Santos
**Dados:** [Inside Airbnb](https://insideairbnb.com/get-the-data/), Rio de Janeiro, coleta de setembro de 2026 (dados públicos)
**Ferramentas:** Python (pandas e matplotlib)

## O objetivo
Sou anfitriã Superhost no Airbnb e quis responder com dados a perguntas que todo anfitrião (ou investidor) se faz:

1. **Quais bairros do Rio têm a diária mais cara?**
2. **Quanto a procura sobe no Réveillon e no Carnaval?**
3. **Superhosts cobram mais caro ou alugam mais?**
4. **Onde vale mais a pena ter um imóvel no Airbnb?**

## Passo 1: carregar os dados
Uso dois arquivos:
- **listings**: uma linha por anúncio (preço, bairro, nota, se o anfitrião é Superhost...)
- **calendar**: uma linha por anúncio **por dia**, dizendo se aquele dia está livre ou ocupado no próximo ano

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

anuncios = pd.read_csv("data/listings.csv.gz", low_memory=False)
calendario = pd.read_csv("data/calendar.csv.gz")

print("Anúncios:", len(anuncios))
print("Linhas do calendário:", len(calendario))

Anúncios: 48247
Linhas do calendário: 17610155


## Passo 2: limpar os dados
Dados reais nunca vêm prontos. Fiz 4 ajustes simples:

1. **Preço vem como texto** ("$1,173.25"). Tiro o "$" e a vírgula e transformo em número. *(Os valores estão em reais; o site usa o símbolo $ para todas as cidades.)*
2. **Anúncios sem preço** saem da análise.
3. **Preços absurdos** (como uma diária de R$ 587 mil) distorcem a média. Removo o 1% mais barato e o 1% mais caro.
4. Fico só com **anúncios ativos** (pelo menos 1 avaliação nos últimos 12 meses) e **imóveis inteiros**, que são 80% do site. Assim comparo coisas parecidas.

In [2]:
# 1. Preço: texto -> número
anuncios["preco"] = (anuncios["price"]
                     .str.replace("$", "", regex=False)
                     .str.replace(",", "", regex=False)
                     .astype(float))

# 2. Remover anúncios sem preço
anuncios = anuncios.dropna(subset=["preco"])

# 3. Remover preços extremos (1% de cada ponta)
limite_baixo = anuncios["preco"].quantile(0.01)
limite_alto = anuncios["preco"].quantile(0.99)
anuncios = anuncios[anuncios["preco"].between(limite_baixo, limite_alto)]
print(f"Faixa de preço mantida: R$ {limite_baixo:.0f} a R$ {limite_alto:.0f}")

# 4. Só anúncios ativos e imóveis inteiros
anuncios = anuncios[(anuncios["number_of_reviews_ltm"] >= 1) &
                    (anuncios["room_type"] == "Entire home/apt")]

# Coluna mais fácil de ler: Superhost Sim/Não
anuncios["superhost"] = anuncios["host_is_superhost"].map({"t": "Sim", "f": "Não"})

print("Anúncios na análise:", len(anuncios))

Faixa de preço mantida: R$ 90 a R$ 7554
Anúncios na análise: 27493


## Pergunta 1: quais bairros têm a diária mais cara?
Uso a **mediana** em vez da média: ela mostra o preço "do meio" e não é puxada por poucos imóveis de luxo.
Só entram bairros com pelo menos 100 anúncios, para a comparação ser justa.

In [3]:
bairros = (anuncios.groupby("neighbourhood_cleansed")
           .agg(anuncios=("id", "count"),
                diaria_mediana=("preco", "median"),
                noites_alugadas=("estimated_occupancy_l365d", "median"),
                faturamento_anual=("estimated_revenue_l365d", "median"))
           .query("anuncios >= 100")
           .sort_values("diaria_mediana", ascending=False))

top10 = bairros.head(10).sort_values("diaria_mediana")
plt.figure(figsize=(8, 5))
plt.barh(top10.index, top10["diaria_mediana"], color="#E0565B")
plt.title("Diária mediana por bairro (top 10)")
plt.xlabel("R$ por noite")
plt.xlim(0, top10["diaria_mediana"].max() * 1.15)
for i, v in enumerate(top10["diaria_mediana"]):
    plt.text(v + 10, i, f"R$ {v:.0f}", va="center")
plt.tight_layout()
plt.show()

bairros.round(0)

                          anuncios  diaria_mediana  noites_alugadas  \
neighbourhood_cleansed                                                
Leblon                        1337           804.0             78.0   
Ipanema                       2552           750.0             90.0   
Lagoa                          123           730.0             36.0   
Gávea                          105           717.0             24.0   
Barra da Tijuca               2030           613.0             66.0   
Leme                           504           508.0             66.0   
Laranjeiras                    254           468.0             42.0   
Copacabana                    9589           460.0             84.0   
Glória                         235           411.0             54.0   
Recreio dos Bandeirantes      1312           410.0             42.0   
Botafogo                      1136           410.0             72.0   
Flamengo                       625           402.0             60.0   
Jacare

![diaria_por_bairro](imagens/diaria_por_bairro.png)

**O que descobri:** Leblon (≈ R$ 800) e Ipanema (≈ R$ 750) têm as diárias mais caras. Copacabana, que tem de longe o maior número de anúncios, cobra quase metade disso (≈ R$ 460).

## Pergunta 2: quanto a procura sobe no Réveillon e no Carnaval?
O arquivo de calendário diz, para cada dia, se o imóvel está **livre** ou **indisponível**. Calculo a **% de imóveis indisponíveis por dia**.

⚠️ *Cuidado na interpretação:* "indisponível" pode ser reserva **ou** o anfitrião bloqueando a data. Por isso olho só os próximos 6 meses (muitos anfitriões fecham as datas distantes) e comparo os picos com os dias normais ao redor.

In [4]:
# Só anúncios que estão na análise
cal = calendario[calendario["listing_id"].isin(anuncios["id"])].copy()
cal["date"] = pd.to_datetime(cal["date"])
cal["ocupado"] = cal["available"] == "f"

# % de imóveis ocupados por dia, de outubro/2026 a março/2027
por_dia = cal.groupby("date")["ocupado"].mean() * 100
por_dia = por_dia["2026-10-01":"2027-03-31"]

plt.figure(figsize=(10, 4))
plt.plot(por_dia.index, por_dia.values, color="#E0565B")
plt.title("% de imóveis indisponíveis por dia no Rio")
plt.ylabel("% indisponível")
plt.annotate("Réveillon", xy=(pd.Timestamp("2026-12-31"), por_dia["2026-12-31"]),
             xytext=(pd.Timestamp("2026-11-20"), 72), arrowprops=dict(arrowstyle="->"))
plt.annotate("Carnaval", xy=(pd.Timestamp("2027-02-07"), por_dia["2027-02-07"]),
             xytext=(pd.Timestamp("2027-02-20"), 72), arrowprops=dict(arrowstyle="->"))
plt.ylim(0, 85)
plt.tight_layout()
plt.show()

print(f"20/dez (dia comum):  {por_dia['2026-12-20']:.0f}% indisponível")
print(f"31/dez (Réveillon):  {por_dia['2026-12-31']:.0f}% indisponível")
print(f"Início de fev:       {por_dia['2027-02-01']:.0f}% indisponível")
print(f"07/fev (Carnaval):   {por_dia['2027-02-07']:.0f}% indisponível")

20/dez (dia comum):  27% indisponível
31/dez (Réveillon):  71% indisponível
Início de fev:       44% indisponível
07/fev (Carnaval):   68% indisponível


![reveillon_carnaval](imagens/reveillon_carnaval.png)

**O que descobri:** em setembro, mais de 3 meses antes, o Réveillon já tinha mais de 70% dos imóveis indisponíveis, mais que o dobro de um dia comum de dezembro. O Carnaval aparece como um segundo pico claro. Para quem é anfitrião, a lição é abrir o calendário e definir o preço dessas datas com muita antecedência.

## Pergunta 3: Superhosts cobram mais caro ou alugam mais?
Comparo os dois grupos em 4 números: diária, nota, noites alugadas no ano e faturamento estimado no ano.

*Observação: "noites alugadas" e "faturamento" são **estimativas** do próprio Inside Airbnb, calculadas a partir das avaliações recebidas.*

In [5]:
comparacao = (anuncios.groupby("superhost")
              .agg(anuncios=("id", "count"),
                   diaria_mediana=("preco", "median"),
                   nota_media=("review_scores_rating", "mean"),
                   noites_alugadas=("estimated_occupancy_l365d", "median"),
                   faturamento_anual=("estimated_revenue_l365d", "median"))
              .round(2))
comparacao

           anuncios  diaria_mediana  nota_media  noites_alugadas  \
superhost                                                          
Não           13436          464.58        4.76             30.0   
Sim           14057          446.00        4.89            108.0   

           faturamento_anual  
superhost                     
Não                  16200.0  
Sim                  48557.0  

In [6]:
fig, eixos = plt.subplots(1, 3, figsize=(11, 4))
grupos = ["Não", "Sim"]
cores = ["#B0B0B0", "#E0565B"]
itens = [("diaria_mediana", "Diária mediana (R$)"),
         ("noites_alugadas", "Noites alugadas/ano"),
         ("faturamento_anual", "Faturamento anual (R$)")]
for eixo, (coluna, titulo) in zip(eixos, itens):
    valores = comparacao.loc[grupos, coluna]
    eixo.bar(["Comum", "Superhost"], valores, color=cores)
    eixo.set_title(titulo)
    eixo.set_ylim(0, valores.max() * 1.15)
    for i, v in enumerate(valores):
        eixo.text(i, v, f"{v:,.0f}".replace(",", "."), ha="center", va="bottom")
plt.suptitle("Superhost x anfitrião comum (imóveis inteiros ativos)")
plt.tight_layout()
plt.show()

![superhost](imagens/superhost.png)

**O que descobri:** o Superhost **não cobra mais caro**: a diária é praticamente igual (≈ R$ 450). A diferença está em **alugar muito mais**: cerca de 108 noites por ano contra 30, o que dá um faturamento estimado **3 vezes maior**.

⚠️ *Correlação não é causa:* para virar Superhost é preciso já ter várias estadias por ano. Então quem aluga mais tende a virar Superhost, e não só o contrário. Mesmo assim, o dado mostra que o selo anda junto com **volume e qualidade** (nota 4,89 contra 4,76), não com preço.

## Pergunta 4: onde vale mais a pena ter um imóvel?
Diária alta não basta: o imóvel também precisa ser alugado. Então olho o **faturamento anual estimado**, que junta as duas coisas (diária × noites alugadas).

In [7]:
top_fat = bairros.sort_values("faturamento_anual", ascending=False).head(10)
top_fat[["anuncios", "diaria_mediana", "noites_alugadas", "faturamento_anual"]].round(0)

                        anuncios  diaria_mediana  noites_alugadas  \
neighbourhood_cleansed                                              
Ipanema                     2552           750.0             90.0   
Leblon                      1337           804.0             78.0   
Barra da Tijuca             2030           613.0             66.0   
Leme                         504           508.0             66.0   
Copacabana                  9589           460.0             84.0   
Lagoa                        123           730.0             36.0   
Botafogo                    1136           410.0             72.0   
Santo Cristo                 431           357.0             72.0   
Flamengo                     625           402.0             60.0   
Glória                       235           411.0             54.0   

                        faturamento_anual  
neighbourhood_cleansed                     
Ipanema                           65416.0  
Leblon                            63315

**O que descobri:** Ipanema e Leblon lideram o faturamento (R$ 63–65 mil por ano), juntando diária alta com muitas noites alugadas. Já bairros como Gávea e Lagoa cobram caro, mas alugam pouco, e acabam faturando menos que Copacabana.

## Conclusão
1. **Localização define o preço:** Leblon e Ipanema cobram quase o dobro de Copacabana.
2. **Datas de pico são reservadas cedo:** Réveillon e Carnaval lotam com meses de antecedência.
3. **Ser Superhost tem a ver com volume, não com preço:** a mesma diária, mas cerca de 3 vezes mais faturamento.
4. **Para investir, olhe o faturamento e não só a diária:** bairro caro com pouca procura rende menos.

**Limitações:** ocupação e faturamento são estimativas; "indisponível" no calendário inclui datas bloqueadas pelo anfitrião; os dados são de uma única coleta (setembro de 2026).

In [8]:
# Salvar bases limpas para montar o dashboard (Power BI / Looker Studio)
colunas = ["id", "neighbourhood_cleansed", "latitude", "longitude", "superhost",
           "accommodates", "bedrooms", "preco", "review_scores_rating",
           "estimated_occupancy_l365d", "estimated_revenue_l365d"]
anuncios[colunas].to_csv("anuncios_limpos.csv", index=False)
bairros.round(2).to_csv("resumo_bairros.csv")
por_dia.round(2).rename("pct_indisponivel").to_csv("ocupacao_por_dia.csv")
print("Arquivos salvos!")

Arquivos salvos!
